In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


# =========================================================
# 1. STATE
# =========================================================

class State(TypedDict):
    count: int


# =========================================================
# 2. NODES
# =========================================================

def add_one(state: State):

    print("ADD_ONE running...")

    return {
        "count": state["count"] + 1
    }


def add_two(state: State):

    print("ADD_TWO running...")

    return {
        "count": state["count"] + 2
    }


def add_three(state: State):

    print("ADD_THREE running...")

    return {
        "count": state["count"] + 3
    }


# =========================================================
# 3. BUILD GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node("add_one", add_one)
builder.add_node("add_two", add_two)
builder.add_node("add_three", add_three)

builder.add_edge(START, "add_one")
builder.add_edge("add_one", "add_two")
builder.add_edge("add_two", "add_three")
builder.add_edge("add_three", END)


# =========================================================
# 4. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 5. ORIGINAL THREAD
# =========================================================

original_config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


print("\n========================================")
print("ORIGINAL THREAD")
print("========================================")


original_result = graph.invoke(
    {
        "count": 0
    },
    config=original_config
)

print(
    "\nOriginal Final Result:",
    original_result
)


# =========================================================
# 6. GET ORIGINAL THREAD HISTORY
# =========================================================

history = list(
    graph.get_state_history(
        original_config
    )
)


print("\n========================================")
print("ORIGINAL CHECKPOINT HISTORY")
print("========================================")


for index, state in enumerate(history):

    checkpoint_id = (
        state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    print(
        f"\nCheckpoint {index}"
    )

    print(
        "Checkpoint ID:",
        checkpoint_id
    )

    print(
        "Values:",
        state.values
    )

    print(
        "Next:",
        state.next
    )


# =========================================================
# 7. SELECT AN OLD CHECKPOINT
# =========================================================

if not history:

    print("No checkpoint history found.")

else:

    # History is newest -> oldest
    old_checkpoint = history[1]

    old_checkpoint_id = (
        old_checkpoint.config[
            "configurable"
        ]["checkpoint_id"]
    )

    old_checkpoint_config = {
        "configurable": {
            "thread_id": "thread-001",
            "checkpoint_id": old_checkpoint_id
        }
    }


    # =====================================================
    # 8. READ OLD CHECKPOINT
    # =====================================================
    old_state = graph.get_state(
        old_checkpoint_config
    )
    print("\n========================================")
    print("SELECTED OLD CHECKPOINT")
    print("========================================")
    print(
        "Thread:",
        old_state.config[
            "configurable"
        ]["thread_id"]
    )
    print(
        "Checkpoint:",
        old_checkpoint_id
    )
    print(
        "Old Values:",
        old_state.values
    )
    print(
        "Old Next:",
        old_state.next
    )
    # =====================================================
    # 9. CREATE NEW THREAD
    # =====================================================

    new_config = {
        "configurable": {
            "thread_id": "thread-002"
        }
    }


    # =====================================================
    # 10. USE OLD STATE AS NEW THREAD'S INPUT
    # =====================================================

    new_initial_state = {
        "count": old_state.values['count']
    }


    print("\n========================================")
    print("NEW THREAD")
    print("========================================")

    print(
        "New Thread:",
        new_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Starting State:",
        new_initial_state
    )


    # =====================================================
    # 11. NEW EXECUTION
    # =====================================================

    new_result = graph.invoke(
        new_initial_state,
        config=new_config
    )


    print("\n========================================")
    print("NEW EXECUTION RESULT")
    print("========================================")

    print(
        "New Result:",
        new_result
    )


    # =====================================================
    # 12. GET NEW THREAD STATE
    # =====================================================

    new_state = graph.get_state(
        new_config
    )


    print("\n========================================")
    print("NEW THREAD CURRENT STATE")
    print("========================================")

    print(
        "Thread:",
        new_state.config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Values:",
        new_state.values
    )

    print(
        "Checkpoint:",
        new_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


    # =====================================================
    # 13. FINAL COMPARISON
    # =====================================================

    print("\n========================================")
    print("COMPARISON")
    print("========================================")

    print(
        "Original Thread ID:",
        original_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Original Final State:",
        original_result
    )

    print()

    print(
        "Old Checkpoint ID:",
        old_checkpoint_id
    )

    print(
        "Old Checkpoint State:",
        old_state.values
    )

    print()

    print(
        "New Thread ID:",
        new_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "New Final State:",
        new_result
    )